In [ ]:
import pandas as pd
import numpy as np
from sentence_transformers import SentenceTransformer
import os
import lightgbm as lgb
from sklearn.datasets import load_svmlight_file
from sklearn.model_selection import train_test_split

In [ ]:
data_path = './data/'

# Baseline submission

In [ ]:
# train_data = pd.read_csv('../data/crowdflower-search-relevance/train_prepared.csv')
test_data = pd.read_csv(os.path.join(data_path, 'crowdflower-search-relevance/test_no_labels.csv'))

# Build a baseline model using query - title match and Catboost ranking model: 
sentences = test_data['query'].tolist() + test_data['product_title'].tolist()

model_sbert = SentenceTransformer("all-MiniLM-L6-v2")
embeddings = model_sbert.encode(sentences, convert_to_numpy=True, show_progress_bar=True)
query_embeddings = embeddings[:len(test_data)]
title_embeddings = embeddings[len(test_data):]

# Baseline (dot product, no training):
scores = np.dot(query_embeddings, title_embeddings.T)
test_data['dot_product_score'] = [scores[i, i] for i in range(len(test_data))]

submission_weak = pd.DataFrame({
    'sample_id': test_data['sample_id'],
    'query_id': test_data['query_id'],
    'score': test_data['dot_product_score']
})
submission_weak.to_csv(os.path.join(data_path, 'crowdflower-search-relevance/baseline_submission.csv'), index=False)

# Learning to rank approach

In [ ]:
def dcg_at_k(y_score, y_true, k):
    order = np.argsort(y_score)[::-1]
    y_true = np.take(y_true, order[:k])
    gain = 2 ** y_true - 1
    discounts = np.log2(np.arange(len(y_true)) + 2)
    return np.sum(gain / discounts)


def ndcg_at_k(y_score, y_true, k):
    dcg_max = dcg_at_k(y_true, y_true, k)
    if not dcg_max:
        return 0.
    return np.round(dcg_at_k(y_score, y_true, k) / dcg_max,4)

In [ ]:
train_data = pd.read_csv(os.path.join(data_path, 'crowdflower-search-relevance/train_prepared.csv'))
train_data.head()

In [ ]:
train_data.value_counts('query_id')

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(train_data, test_size = 0.1, stratify=['query_id'])